In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

In [ ]:
mct = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.passqc_nuclei.h5ad')
multiome = sc.read_h5ad(f'{projdir}/h5ad/raw/multiome.h5ad')

In [ ]:
mct

In [ ]:
multiome

In [ ]:
mct.X[:100,:100].toarray()

In [ ]:
multiome.X[:100,:100].toarray()

In [ ]:
sum(mct.var.index == multiome.var.index)

In [ ]:
sc.pp.filter_cells(mct, min_genes=200)
sc.pp.filter_genes(mct, min_cells=10)

In [ ]:
sc.pp.filter_cells(multiome, min_genes=200)
sc.pp.filter_genes(multiome, min_cells=10)

In [ ]:
mct

In [ ]:
multiome

In [ ]:
mct_mask = mct.var.index.isin(multiome.var.index)
multiome_mask = multiome.var.index.isin(mct.var.index)
final_mct = mct[:, mct_mask]
final_multiome = multiome[:, multiome_mask]
final_mct.shape, final_multiome.shape

In [ ]:
np.sum(final_mct.var.index == final_multiome.var.index) == final_mct.shape[1]

In [ ]:
final_mct.X

In [ ]:
final_multiome.X

In [ ]:
final_multiome.obs.rename(columns={'assignment' : 'donor'}, inplace=True)

In [ ]:
final_multiome.obs['donor'].head(), \
final_mct.obs['donor'].head(), 

### process individually and merge at the gene level

In [ ]:
n_top_genes = 3000

In [ ]:
### mct
sc.pp.normalize_total(final_mct, target_sum=1e4,
                      exclude_highly_expressed=True)
sc.pp.log1p(final_mct)
sc.pp.scale(final_mct, max_value=10)

In [ ]:
final_mct

In [ ]:
final_mct.X

In [ ]:
### multiome
sc.pp.normalize_total(final_multiome, target_sum=1e4,
                      exclude_highly_expressed=True)
sc.pp.log1p(final_multiome)
sc.pp.scale(final_multiome, max_value=10)

In [ ]:
final_multiome

In [ ]:
final_multiome.X

In [ ]:
final_mct.obs['assay'] = 'snmct'
final_multiome.obs['assay'] = 'multiome'

In [ ]:
merged = anndata.concat([final_mct, final_multiome])

In [ ]:
merged

In [ ]:
merged.X

In [ ]:
sc.pp.highly_variable_genes(merged, n_top_genes=n_top_genes)

In [ ]:
# merged.var.to_csv(f'{projdir}/csv/rna/merged_hvgs_all.csv', sep='\t', header=True, index=True)

In [ ]:
n_comps = 50
sc.tl.pca(merged, n_comps=n_comps)

In [ ]:
merged.obsm['X_pca'].shape

In [ ]:
sc.pl.pca(merged, components=['1,2','3,4','5,6','7,8'],
          wspace=0.3, color = ['assay', 'time_num', 'donor'])

In [ ]:
tmp_merged = merged[merged.obs['assay'] == 'snmct']
sc.pl.pca(tmp_merged, components=['1,2','3,4','5,6','7,8'],
          wspace=0.3, color = ['assay', 'time_num', 'donor'])

In [ ]:
tmp_merged = merged[merged.obs['assay'] == 'multiome']
sc.pl.pca(tmp_merged, components=['1,2','3,4','5,6','7,8'],
          wspace=0.3, color = ['assay', 'time_num', 'donor'])

In [ ]:
# sc.pl.pca(adata, components=['1,2','3,4','5,6','7,8',
#                             '9,10','11,12','13,14',
#                              '15,16', '17,18','19,20'],
#           wspace=0.3)

In [ ]:
to_keep = 20
sc.pp.neighbors(merged, random_state=0, n_pcs=to_keep)
sc.tl.leiden(merged, resolution=0.5, random_state=0)
sc.tl.umap(merged, random_state=0)

In [ ]:
sc.pl.umap(merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'snmct']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'multiome']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

In [ ]:
# cols = [f'PC{x}' for x in range(1, 51)]
# merged_pcs = pd.DataFrame(merged.obsm['X_pca'], columns = cols, index=merged.obs.index)
# merged_pcs.to_csv(f'{projdir}/csv/integrate/merged_gene_level.pcs.csv', sep='\t', header=True, index=True)

# cols = [f'umap{x}' for x in range(1, 3)]
# merged_umap = pd.DataFrame(merged.obsm['X_umap'], columns = cols, index=merged.obs.index)
# merged_umap.to_csv(f'{projdir}/csv/integrate/merged_gene_level.umap.csv', sep='\t', header=True, index=True)

### harmony

In [ ]:
sc.external.pp.harmony_integrate(merged, key = 'assay')

In [ ]:
merged.obsm['X_pca']

In [ ]:
merged.obsm['X_pca_harmony']

In [ ]:
sc.pp.neighbors(merged, random_state=0, n_pcs=to_keep, use_rep='X_pca_harmony')
sc.tl.leiden(merged, resolution=0.5, random_state=0)
sc.tl.umap(merged, random_state=0)

In [ ]:
sc.pl.umap(merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'snmct']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'multiome']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

In [ ]:
# ###
# cols = [f'PC{x}' for x in range(1, 51)]
# merged_pcs = pd.DataFrame(merged.obsm['X_pca_harmony'], columns = cols, index=merged.obs.index)
# merged_pcs.to_csv(f'{projdir}/csv/integrate/merged_gene_level_harmony.all_nuclei.pcs.csv', sep='\t', header=True, index=True)

# cols = [f'umap{x}' for x in range(1, 3)]
# merged_umap = pd.DataFrame(merged.obsm['X_umap'], columns = cols, index=merged.obs.index)
# merged_umap.to_csv(f'{projdir}/csv/integrate/merged_gene_level_harmony..all_nuclei.umap.csv', sep='\t', header=True, index=True)

### now do a per-donor harmony

In [ ]:
n_pcs = 20

In [ ]:
for donor in donors:
    print(donor)
    donor_merged = merged[merged.obs['donor'] == donor]
    sc.external.pp.harmony_integrate(donor_merged, key = 'assay')
    sc.pp.neighbors(donor_merged, random_state=0, n_pcs=n_pcs, use_rep='X_pca_harmony')
    sc.tl.leiden(donor_merged, resolution=0.5, random_state=0)
    sc.tl.umap(donor_merged, random_state=0)

    sc.pl.umap(donor_merged, color = ['assay', 'time_num', 'donor'])

    tmp_donor_merged = donor_merged[donor_merged.obs['assay'] == 'snmct']
    sc.pl.umap(tmp_donor_merged, color = ['assay', 'time_num', 'donor'])

    tmp_donor_merged = donor_merged[donor_merged.obs['assay'] == 'multiome']
    sc.pl.umap(tmp_donor_merged, color = ['assay', 'time_num', 'donor'])
    
    cols = [f'PC{x}' for x in range(1, 51)]
    donor_merged_pcs = pd.DataFrame(donor_merged.obsm['X_pca_harmony'], columns = cols, index=donor_merged.obs.index)
    donor_merged_pcs.to_csv(f'{projdir}/csv/integrate/per_donor/{donor}.passqc_nuclei.pcs.csv', sep='\t', header=True, index=True)

    cols = [f'umap{x}' for x in range(1, 3)]
    donor_merged_umap = pd.DataFrame(donor_merged.obsm['X_umap'], columns = cols, index=donor_merged.obs.index)
    donor_merged_umap.to_csv(f'{projdir}/csv/integrate/per_donor/{donor}.passqc_nuclei.umap.csv', sep='\t', header=True, index=True)

### and finally, let's merge all of the per-donor harmony corrected PCs together and make a umap

In [ ]:
donor_pc_list = []
for donor in donors:
    print(donor)
    tmp_donor_pcs = pd.read_csv(f'{projdir}/csv/integrate/per_donor/{donor}.passqc_nuclei.pcs.csv', sep='\t', header=0, index_col=0)
    donor_pc_list.append(tmp_donor_pcs)
donor_pcs = pd.concat(donor_pc_list)

In [ ]:
merged.obsm['X_pca_donor_merged_harmony'] = donor_pcs.reindex(merged.obs.index).values

In [ ]:
n_pcs = 20
sc.pp.neighbors(merged, random_state=0, n_pcs=n_pcs, use_rep='X_pca_donor_merged_harmony')
sc.tl.leiden(merged, resolution=0.5, random_state=0)
sc.tl.umap(merged, random_state=0)

In [ ]:
sc.pl.umap(merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'snmct']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

tmp_merged = merged[merged.obs['assay'] == 'multiome']
sc.pl.umap(tmp_merged, color = ['assay', 'time_num', 'donor'])

In [ ]:
###
cols = [f'PC{x}' for x in range(1, 51)]
merged_pcs = pd.DataFrame(merged.obsm['X_pca_donor_merged_harmony'], columns = cols, index=merged.obs.index)
merged_pcs.to_csv(f'{projdir}/csv/integrate/merged_per_donor_harmony.passqc_nuclei.pcs.csv', sep='\t', header=True, index=True)

cols = [f'umap{x}' for x in range(1, 3)]
merged_umap = pd.DataFrame(merged.obsm['X_umap'], columns = cols, index=merged.obs.index)
merged_umap.to_csv(f'{projdir}/csv/integrate/merged_per_donor_harmony.passqc_nuclei.umap.csv', sep='\t', header=True, index=True)

In [ ]:
merged.write_h5ad(f'{projdir}/h5ad/processed/merged.passqc_nuclei.h5ad')

In [ ]:
!date